# B1 ? Formula-Based A-S Parameter PPO

**Purpose:** Train the unconstrained RL market maker as an Alpha-AS-style parameter policy.

The PPO action is no longer a direct quote depth. It is `[u_gamma, u_skew]`; the environment maps this to `(gamma_t, skew_t)`, computes A-S quotes, then passes the resulting depths to the same fill simulator used by B0. Market parameters `sigma`, `A`, and `kappa` are calibrated on training days 1-6 only.


In [ ]:
import sys
import pathlib
import shutil
from dataclasses import replace

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

PROJECT_ROOT = next(
    (p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents]
     if (p / "procs").exists()),
    pathlib.Path.cwd(),
)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from stable_baselines3 import PPO

from procs.gym.experiment_config import ReplayExperimentConfig
from procs.gym.data_loader import load_multi_day_with_features
from procs.gym.calibration import calibrate_from_arrays
from procs.gym.formula_as import FORMULA_FEATURE_NAMES
from procs.gym.sb3_wrapper import StableBaselinesTradingEnvironment
from procs.gym.notebook_support import (
    build_formula_multi_day_replay_env,
    make_vecnorm,
    evaluate_formula_rl_per_day,
)
from procs.rewards import CjMmCriterion

cfg = ReplayExperimentConfig()
cfg.ensure_artifact_dirs()
print(f"Repo root : {cfg.repo_root}")
print(f"Models    : {cfg.models_dir}")
print(f"Results   : {cfg.results_dir}")


## Section 1 — Data Loading and Train/Test Split

In [ ]:
TRAIN_DAYS = 6

daily_S, daily_dt, dates, daily_features = load_multi_day_with_features(
    str(cfg.datasets_dir),
    pair=cfg.pair,
    tick_size=cfg.tick_size,
)

train_S        = daily_S[:TRAIN_DAYS]
train_dt       = daily_dt[:TRAIN_DAYS]
train_dates    = dates[:TRAIN_DAYS]
train_features = daily_features[:TRAIN_DAYS]

test_S         = daily_S[TRAIN_DAYS:]
test_dt        = daily_dt[TRAIN_DAYS:]
test_dates     = dates[TRAIN_DAYS:]
test_features  = daily_features[TRAIN_DAYS:]

EXPECTED_TEST_DAYS = 23
if len(train_S) != TRAIN_DAYS or len(test_S) != EXPECTED_TEST_DAYS:
    raise ValueError(
        f"Expected {TRAIN_DAYS} train days and {EXPECTED_TEST_DAYS} test days, "
        f"found {len(train_S)} and {len(test_S)}."
    )
if set(map(str, train_dates)) & set(map(str, test_dates)):
    raise ValueError("Train/test date overlap detected.")

# Train-only market calibration: these values stay fixed for all test days.
param_rows = []
for date, S, dt in zip(train_dates, train_S, train_dt):
    sigma, A, kappa = calibrate_from_arrays(S, dt, tick_size=cfg.tick_size)
    param_rows.append({"Day": date, "sigma": sigma, "A": A, "kappa": kappa})

df_train_params = pd.DataFrame(param_rows).set_index("Day")
sigma_train = float(df_train_params["sigma"].median())
A_train = float(df_train_params["A"].median())
kappa_train = float(df_train_params["kappa"].median())
cfg = replace(cfg, A=A_train, kappa=kappa_train)

FORMULA_KWARGS = dict(
    sigma=sigma_train,
    gamma_min=0.01,
    gamma_max=0.9,
    skew_ticks_max=5.0,
)

train_snapshots = sum(len(S) for S in train_S)

print(f"Training days : {len(train_S)}  ({train_dates[0]} -> {train_dates[-1]})")
print(f"Test days     : {len(test_S)}  ({test_dates[0]} -> {test_dates[-1]})")
print(f"Training snapshots: {train_snapshots:,} (~{train_snapshots/1e6:.2f}M)")
print()
print("Train-only A-S market parameters (median over days 1-6):")
print(f"  sigma = {sigma_train:.8f}")
print(f"  A     = {A_train:.6f}")
print(f"  kappa = {kappa_train:.2f}")
print(f"Formula features: {list(FORMULA_FEATURE_NAMES)}")


## Section 2 — Build Multi-Day Training Environment

In [ ]:
reward_fn_train = CjMmCriterion(per_step_inventory_aversion=cfg.phi)

train_env = build_formula_multi_day_replay_env(
    train_S,
    train_dt,
    cfg,
    daily_market_features=train_features,
    reward_fn=reward_fn_train,
    mode="sequential",
    **FORMULA_KWARGS,
)
train_sb3 = StableBaselinesTradingEnvironment(train_env)
train_vn = make_vecnorm(train_sb3, cfg, training=True, norm_reward=True)

obs_dim = train_env.observation_space.shape[0]
print(f"Observation dim : {obs_dim}  ({list(FORMULA_FEATURE_NAMES)})")
print(f"Action dim      : {train_env.action_space.shape[0]}  ([u_gamma, u_skew])")
print(f"Action bounds   : gamma in [0.01, 0.9], skew in +/-5 ticks")
print(f"Training days   : {len(train_S)} separate replay episodes")
print(f"Snapshots total : {train_snapshots:,}")


## Section 3 — PPO Training

**`TOTAL_TIMESTEPS` guide:**
- Quick local test: `200_000`
- Full Snellius run: `max(2 * len(S_train), 2_000_000)`

In [ ]:
TOTAL_TIMESTEPS = max(train_snapshots, 1_000_000)  # ≥1 full pass or 1M steps
print(f"TOTAL_TIMESTEPS = {TOTAL_TIMESTEPS:,}  ({TOTAL_TIMESTEPS/train_snapshots:.2f}× data)")

model_b1 = PPO(
    "MlpPolicy",
    train_vn,
    **cfg.ppo_kwargs(),
    tensorboard_log=str(cfg.repo_root / "tb_logs" / "b1"),
    verbose=1,
    device="cpu",
)

print(f"PPO kwargs: {cfg.ppo_kwargs()}")
model_b1.learn(total_timesteps=TOTAL_TIMESTEPS)


In [ ]:
model_zip = cfg.model_path("ppo_b1_doge").with_suffix(".zip")
model_archive = cfg.model_path("ppo_b1_doge_direct_depth").with_suffix(".zip")
if model_zip.exists() and not model_archive.exists():
    shutil.copy2(model_zip, model_archive)
    print(f"Archived previous direct-depth model -> {model_archive}")

vecnorm_path = cfg.vecnorm_path("vecnorm_b1")
vecnorm_archive = cfg.vecnorm_path("vecnorm_b1_direct_depth")
if vecnorm_path.exists() and not vecnorm_archive.exists():
    shutil.copy2(vecnorm_path, vecnorm_archive)
    print(f"Archived previous direct-depth VecNormalize -> {vecnorm_archive}")

model_b1.save(str(cfg.model_path("ppo_b1_doge")))
train_vn.save(str(cfg.vecnorm_path("vecnorm_b1")))
print(f"Saved formula-AS model   -> {cfg.model_path('ppo_b1_doge')}.zip")
print(f"Saved formula-AS vecnorm -> {cfg.vecnorm_path('vecnorm_b1')}")


## Section 4 — Training Diagnostics

Quick diagnostic from the last logged progress (PPO prints every `n_steps` rollout).
For full curves, run: `tensorboard --logdir tb_logs/b1`

In [ ]:
# Parse TensorBoard event file for explained variance and entropy
try:
    from tensorflow.python.summary.summary_iterator import summary_iterator
    tb_dir = cfg.repo_root / "tb_logs" / "b1"
    ev_files = sorted(tb_dir.glob("events.out.tfevents.*"))
    if ev_files:
        ev_data = {"step": [], "explained_variance": [], "entropy_loss": []}
        for event in summary_iterator(str(ev_files[-1])):
            for v in event.summary.value:
                if v.tag == "train/explained_variance":
                    ev_data["step"].append(event.step)
                    ev_data["explained_variance"].append(v.simple_value)
                elif v.tag == "train/entropy_loss":
                    ev_data["entropy_loss"].append(v.simple_value)

        df_tb = pd.DataFrame(ev_data).dropna()
        if len(df_tb):
            fig, axes = plt.subplots(1, 2, figsize=(12, 4))
            axes[0].plot(df_tb["step"], df_tb["explained_variance"])
            axes[0].axhline(0.5, color="red", linestyle="--", alpha=0.5, label="target ≥ 0.5")
            axes[0].set_title("Explained Variance")
            axes[0].legend(); axes[0].grid(True, alpha=0.3)
            axes[1].plot(df_tb["step"], df_tb.get("entropy_loss", []))
            axes[1].set_title("Entropy Loss")
            axes[1].grid(True, alpha=0.3)
            plt.suptitle("B1 Training Diagnostics")
            plt.tight_layout()
            plt.show()
        print(f"Final explained variance: {df_tb['explained_variance'].iloc[-1]:.4f}")
    else:
        print("No TensorBoard event files found — skipping diagnostic plot.")
except Exception as e:
    print(f"TensorBoard parsing unavailable ({e}). Run: tensorboard --logdir tb_logs/b1")


## Section 5 — Test Evaluation (Days 7–29)

In [ ]:
model_b1_loaded = PPO.load(str(cfg.model_path("ppo_b1_doge")), device="cpu")

df_b1 = evaluate_formula_rl_per_day(
    model=model_b1_loaded,
    vecnorm_path=cfg.vecnorm_path("vecnorm_b1"),
    test_S=test_S,
    test_dt=test_dt,
    test_dates=test_dates,
    test_market_features=test_features,
    config=cfg,
    seed=cfg.evaluation_seed,
    num_rollouts=cfg.evaluation_rollouts,
    **FORMULA_KWARGS,
)

print(f"Evaluated {len(df_b1)} test days with {cfg.evaluation_rollouts} rollouts per day.")


## Section 6 — Results Display and Save

In [ ]:
pd.set_option("display.float_format", "{:.4f}".format)
METRICS = ["Sharpe", "Sortino", "Max DD", "P&L-to-MAP", "Final PnL"]

summary = pd.DataFrame({
    "Mean":   df_b1[METRICS].mean(),
    "Std":    df_b1[METRICS].std(),
    "Median": df_b1[METRICS].median(),
}).T

print("=== B1 Per-day Test Results ===")
print(df_b1[METRICS].to_string())
print("\n=== B1 Summary ===")
print(summary.to_string())


In [ ]:
out_path = cfg.result_path("b1_test_results.csv")
archive_path = cfg.result_path("b1_direct_depth_test_results.csv")
if out_path.exists() and not archive_path.exists():
    shutil.copy2(out_path, archive_path)
    print(f"Archived previous direct-depth results -> {archive_path}")

df_b1.to_csv(out_path)
print(f"Saved formula-AS B1 results -> {out_path}")
